# 🏟️ Práctica Bases de Datos NoSQL — Análisis de Jugadores para Director Deportivo

**Dataset:** [European Top Football Leagues Player Stats](https://www.kaggle.com/datasets/kaanyorgun/european-top-leagues-player-stats-25-26?select=all_player_stats.csv) — Kaan Yorgun  
**Tecnología:** MongoDB + PyMongo  
**Contexto:** Nos ponemos en la piel del director deportivo de un club de fútbol. Con un presupuesto limitado de 500M€, debemos identificar los mejores jugadores por posición optimizando rendimiento y valor de mercado.

---

## Estructura del notebook

| Sección | Contenido |
|---|---|
| 1 | Conexión y configuración |
| 2 | Carga e ingesta de datos |
| 3 | Queries básicas de exploración |
| 4 | Aggregation Pipelines avanzados |
| 5 | Construcción del equipo óptimo |

Primero, comenzamos la práctica instalando la librería de pymongo si no la tenemos instalada previamente

In [1]:
pip install pymongo


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


A continuación, iniciamos la conexión con el servidor en local de MongoDB:

In [2]:
import pymongo

# La URI para local siempre es esta:
uri_local = "mongodb://localhost:27017/"

try:
    client = pymongo.MongoClient(uri_local)
    # Verificamos si el servidor responde
    client.admin.command('ping')
    print("✅  ¡Conectado exitosamente al MongoDB local!")
except Exception as e:
    print(f"Error al conectar: {e}")

✅  ¡Conectado exitosamente al MongoDB local!


Una vez conectado al servidor de MongoDB, nuestro trabajo será ponernos en la piel del Director Deportivo de un equipo de fútbol para tratar de crear el mejor equipo posible basado en las estadísticas qeu tenemos disponibles. Para ello, nos descargaremos el dataset con las estadísticas de la presente temporada de las ligas europeas. 

In [3]:
# Este código ha sido importado directamente desde Kaggle para descargar el dataset.

import kagglehub

# Download latest version
DATA_PATH = kagglehub.dataset_download("kaanyorgun/european-top-leagues-player-stats-25-26")

print("Path to dataset files:", DATA_PATH)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/carlos/.cache/kagglehub/datasets/kaanyorgun/european-top-leagues-player-stats-25-26/versions/11


El dataset que obtenemos al ejecutar el cuadro anterior está compuesto por dos CSVs:

- all_player_profiles.csv
- all_player_stats.csv

En nuestro caso, no estamos interesados en trabajar con CSVs, luego lo transformaremos a JSON y aunque MongoDB  no esté pensado para ello, haremos un left join para unir ambos ficheros y obtener la totalidad de los datos limpios listos para trabajar. 

In [4]:
import csv
import json

print("📂 Cargando CSV y transformando a JSON...")

# 1. Abrimos el archivo CSV para leer
with open(DATA_PATH + "/all_player_profiles.csv", mode='r', encoding='utf-8') as csv_file:
    # DictReader convierte cada fila del CSV en un diccionario, usando la cabecera como claves
    reader = csv.DictReader(csv_file)
    datos = list(reader) # Lo convertimos en una lista en memoria

# 2. Guardamos esa lista de diccionarios en un archivo JSON
with open(DATA_PATH + "/all_player_profiles.json", mode='w', encoding='utf-8') as json_file:
    # json.dump escribe los datos en el archivo. indent=4 lo hace legible (pretty print)
    json.dump(datos, json_file, indent=4)

# 3. Abrimos el archivo CSV para leer (Ahora el archivo de estadísticas)
with open(DATA_PATH + "/all_player_stats.csv", mode='r', encoding='utf-8') as csv_file:
    # DictReader convierte cada fila del CSV en un diccionario, usando la cabecera como claves
    reader = csv.DictReader(csv_file)
    datos = list(reader) # Lo convertimos en una lista en memoria

# 4. Guardamos esa lista de diccionarios en un archivo JSON
with open(DATA_PATH + "/all_player_stats.json", mode='w', encoding='utf-8') as json_file:
    # json.dump escribe los datos en el archivo. indent=4 lo hace legible (pretty print)
    json.dump(datos, json_file, indent=4)

print("✅  CSVs cargados y transformados a JSON exitosamente.")

📂 Cargando CSV y transformando a JSON...
✅  CSVs cargados y transformados a JSON exitosamente.


In [5]:
# Borrar la base de datos por nombre si existiese (opcional, para evitar conflictos al volver a ejecutar el código)
client.drop_database('football_management')
print("Base de datos eliminada.")

Base de datos eliminada.


Ya tenemos los datos transformados y listos para trabajar. Ahora, subimos al cliente los archivos json.

In [6]:
INT_FIELDS = ['age', 'market_value', 'appearances', 'matches_started',  'minutes_played', 'goals', 'assists', 'saves', 'expected_goals', 'expected_assists', 'yellow_cards', 'red_cards', 'total_shots', 'shots_on_target', 'interceptions', 'tackles']

# 1. Configurar la conexión
try:
    db = client["football_management"]
    players = db["players"]
    datos = db["stats"]
    
    # 2. Cargar el archivo JSON
    with open(DATA_PATH + "/all_player_profiles.json", 'r', encoding='utf-8') as f:
        datos_json_players = json.load(f)

    with open(DATA_PATH + "/all_player_stats.json", 'r', encoding='utf-8') as f:
        datos_json_stats = json.load(f)
    
    # Convertimos campos numéricos a int/float para queries correctas
    for doc in datos_json_players:
        if 'market_value' in doc and doc['market_value'] is not None:
            try:
                doc['market_value'] = int(doc['market_value'])
            except ValueError:
                pass  # si no es convertible, dejar como está
    
    # Convertimos campos numéricos a int/float para queries correctas
    for doc in datos_json_stats:
        for campo in INT_FIELDS:
            if campo in doc and doc[campo] is not None:
                try:
                    doc[campo] = int(doc[campo])
                except ValueError:
                    pass  # si no es convertible, dejar como está
            
            if campo == 'rating' and doc['rating'] is not None:
                try:
                    doc['rating'] = float(doc['rating'])
                except ValueError:
                    pass  # si no es convertible, dejar como está


    # 3. Insertamos los datos
    insertar_jugadores = players.insert_many(datos_json_players)
    insertar_stats = datos.insert_many(datos_json_stats)

    print(f"✅ Se han insertado {len(insertar_jugadores.inserted_ids)} jugadores correctamente.")
    print(f"✅ Se han insertado {len(insertar_stats.inserted_ids)} estadísticas correctamente.")

except Exception as e:
    print(f"❌ Ocurrió un error inesperado: {e}")
except FileNotFoundError:
    print("❌ Error: No se encontró el archivo 'all_player_profiles.json'.")

✅ Se han insertado 4203 jugadores correctamente.
✅ Se han insertado 3646 estadísticas correctamente.


In [7]:
# 1. Ver UN solo documento (el primero que encuentre)
print("--- Un solo jugador ---")
jugador = players.find_one()
print(jugador)

# 2. Ver VARIOS documentos con un filtro 
print("\n--- Jugadores Estrella ---")
query = {"market_value": {"$gt": 10000}}
for doc in players.find(query).limit(3):  # limit para no spam
    print(f"Nombre: {doc['name']} | Valor: {doc['market_value']} €")

# 3. Ver TODOS (limitado para no colapsar la consola)
print("\n--- Listado general (Top 5) ---")
for doc in players.find().limit(5):
    print(doc)

--- Un solo jugador ---
{'_id': ObjectId('69d231ee38c920497f4cd43c'), 'player_id': '804508', 'name': 'Viktor Gyökeres', 'league': 'Premier League', 'position': 'F', 'market_value': 61000000}

--- Jugadores Estrella ---
Nombre: Viktor Gyökeres | Valor: 61000000 €
Nombre: Bukayo Saka | Valor: 126000000 €
Nombre: Gabriel Jesus | Valor: 21000000 €

--- Listado general (Top 5) ---
{'_id': ObjectId('69d231ee38c920497f4cd43c'), 'player_id': '804508', 'name': 'Viktor Gyökeres', 'league': 'Premier League', 'position': 'F', 'market_value': 61000000}
{'_id': ObjectId('69d231ee38c920497f4cd43d'), 'player_id': '934235', 'name': 'Bukayo Saka', 'league': 'Premier League', 'position': 'F', 'market_value': 126000000}
{'_id': ObjectId('69d231ee38c920497f4cd43e'), 'player_id': '794839', 'name': 'Gabriel Jesus', 'league': 'Premier League', 'position': 'F', 'market_value': 21000000}
{'_id': ObjectId('69d231ee38c920497f4cd43f'), 'player_id': '922573', 'name': 'Gabriel Martinelli', 'league': 'Premier League'

Como estamos interesados en la relación de las estadísticas con los jugadores que tenemos guardados en la colección players, utilizaremos el pipeliine de  agregación:

In [8]:
# Definimos el pipeline de agregación
pipeline = [
    {
        "$lookup": {
            "from": "stats",             # La colección con la que queremos unir
            "localField": "player_id",     # El campo en la colección 'players'
            "foreignField": "player_id",   # El campo en la colección 'stats'
            "as": "stats_del_jugador"        # Nombre del nuevo campo donde se guardará el resultado
        }
    } ,
    {
        # Opcional: $unwind convierte la lista 'stats_del_jugador' en un objeto simple
        "$unwind": "$stats_del_jugador"
    },
   
]

# Ejecutar la consulta
resultados = db.players.aggregate(pipeline)

for res in resultados:
    print(res)

{'_id': ObjectId('69d231ee38c920497f4cd43c'), 'player_id': '804508', 'name': 'Viktor Gyökeres', 'league': 'Premier League', 'position': 'F', 'market_value': 61000000, 'stats_del_jugador': {'_id': ObjectId('69d231ef38c920497f4ce4a7'), 'player_id': '804508', 'league': 'Premier League', 'appearances': 29, 'matches_started': 23, 'minutes_played': 1907, 'goals': 11, 'assists': 0, 'expected_goals': '9.3498', 'expected_assists': '1.76750513', 'rating': '6.5724137931034', 'total_shots': 43, 'shots_on_target': 17, 'yellow_cards': 4, 'red_cards': 0, 'tackles': 6, 'interceptions': 1, 'saves': 0}}
{'_id': ObjectId('69d231ee38c920497f4cd43d'), 'player_id': '934235', 'name': 'Bukayo Saka', 'league': 'Premier League', 'position': 'F', 'market_value': 126000000, 'stats_del_jugador': {'_id': ObjectId('69d231ef38c920497f4ce4a8'), 'player_id': '934235', 'league': 'Premier League', 'appearances': 27, 'matches_started': 22, 'minutes_played': 2001, 'goals': 6, 'assists': 3, 'expected_goals': '6.9941', 'expe

In [9]:
# Hacemos un count para comprobar que se ha mezclado correctamente 
# (debería ser el mismo número de jugadores que antes, pero ahora 
# con un campo extra 'stats_del_jugador')
print(db.players.count_documents({}))

4203


Comenzamos con las queries para formar nuestro equipo:

In [10]:
# Comenzamos a seleccionar nuestros jugadores. Comenzamos por la portería, siendo el portero que más nos interesa 
# aquel que tenga mayor proporción de paradas por partido. Para ello, usamos aggregation pipeline.

portero_pipeline = [
    {
        "$lookup": {
            "from": "stats",
            "localField": "player_id",
            "foreignField": "player_id",
            "as": "stats_del_jugador"
        }
    },
    {
        "$unwind": "$stats_del_jugador"
    },
    {
        "$match": {
            "position": "G",
            "stats_del_jugador.appearances": {"$gt": 10},
            "stats_del_jugador.saves": {"$gt": 0}
        }
    },
    {   
        # Creamos un nuevo campo 'save_ratio' que es el resultado de dividir saves entre appearances. Nos interesa este ratio para comparar porteros 
        # que hayan jugado un número elevado de partidos y que realicen muchas paradas para  asegura nuestra portería.
        "$addFields": {
            "save_ratio": {"$divide": ["$stats_del_jugador.saves", "$stats_del_jugador.appearances"]}
        }
    },
    {
        "$sort": {"save_ratio": -1}  # Mayor saves primero
    },
    {
        "$limit": 5  # Limitar a los top 5 para tener opciones y no solo uno
    }
]

portero = list(db.players.aggregate(portero_pipeline))
for player in portero:
    print(f"Nombre: {player['name']} | Saves: {player['stats_del_jugador']['saves']} | Appearances: {player['stats_del_jugador']['appearances']} | Market Value: {player['market_value']}")
if portero:
    GK = portero[0]
    print(f"🏆 Portero seleccionado: {GK['name']} | Saves: {GK['stats_del_jugador']['saves']} | Appearances: {GK['stats_del_jugador']['appearances']} | Market Value: {GK['market_value']}")
else:
    print("❌ No se encontró portero que cumpla los criterios")

Nombre: Kayne van Oevelen | Saves: 134 | Appearances: 28 | Market Value: 2100000
Nombre: Aarón Escandell | Saves: 127 | Appearances: 29 | Market Value: 1900000
Nombre: Jari De Busser | Saves: 122 | Appearances: 28 | Market Value: 3200000
Nombre: Tom de Graaff | Saves: 112 | Appearances: 26 | Market Value: 875000
Nombre: Ronald Koeman Jr | Saves: 116 | Appearances: 27 | Market Value: 595000
🏆 Portero seleccionado: Kayne van Oevelen | Saves: 134 | Appearances: 28 | Market Value: 2100000


In [ ]:
# Ahora pasamos a por nuestro número 2. Nos interesa un lateral derecho defensivo ya que construiremos un equipo 
# atacante por la  izquierda, luego  será el lateral derecho el que bascule para hacer balance defensivo.

lateral_pipeline = [
    {
        "$lookup": {
            "from": "stats",
            "localField": "player_id",
            "foreignField": "player_id",
            "as": "stats_del_jugador"
        }
    },
    {
        "$unwind": "$stats_del_jugador"
    },
    {
        "$match": {
            "position": "D",
            # Aunque el lateral derecho es un jugador defensivo, nos interesa que tenga una buena capacidad de ataque 
            # para que pueda sumarse a las jugadas ofensivas y aportar asistencias. Por eso, filtramos por aquellos laterales 
            # derechos que tengan asistencias, lo que indica que generan oportunidades de gol para sus compañeros.
            "stats_del_jugador.assists": {"$gt": 1},
            "stats_del_jugador.tackles": {"$gt": 10}
        }
    },

    {
        "$sort": {"market_value": 1}  # Menor precio primero
    },
    {
        "$limit": 5  # Limitar a los top 5 para tener opciones y no solo uno
    }
]

lateral = list(db.players.aggregate(lateral_pipeline))
for player in lateral:
    print(f"Nombre: {player['name']} | Tackles: {player['stats_del_jugador']['tackles']} | Assists: {player['stats_del_jugador']['assists']} | Market Value: {player['market_value']}")
if lateral:
    LD = lateral[0]
    print(f"🏆 Lateral seleccionado: {LD['name']} | Tackles: {LD['stats_del_jugador']['tackles']} | Assists: {LD['stats_del_jugador']['assists']} | Market Value: {LD['market_value']}")
else:
    print("❌ No se encontró lateral que cumpla los criterios")

Nombre: Ivo Pinto | Tackles: 44 | Assists: 5 | Market Value: 260000
Nombre: Patrick van Aanholt | Tackles: 18 | Assists: 3 | Market Value: 290000
Nombre: Isaac James | Tackles: 25 | Assists: 2 | Market Value: 340000
Nombre: Ahmet Oğuz | Tackles: 55 | Assists: 2 | Market Value: 380000
Nombre: Hennes Behrens | Tackles: 17 | Assists: 2 | Market Value: 410000
🏆 Lateral seleccionado: Ivo Pinto | Tackles: 44 | Assists: 5 | Market Value: 260000


In [ ]:
# A continuación, buscamos fichar a nuestros dos centrales. Queremos centrales con buen número de intercepciones
# y pocas tarjetas para asegurar un buen rendimiento defensivo sin perder agresividad. Para ello, filtramos por 
# aquellos centrales que tengan un número elevado de intercepciones y un número bajo de tarjetas amarillas y rojas. 
# Además, nos interesa que tengan un precio asequible para no gastar demasiado presupuesto en esta posición.

defensas_pipeline = [
    {
        "$lookup": {
            "from": "stats",
            "localField": "player_id",
            "foreignField": "player_id",
            "as": "stats_del_jugador"
        }
    },
    {
        "$unwind": "$stats_del_jugador"
    },
    {
        "$match": {
            "position": "D",
            "stats_del_jugador.interceptions": {"$gt": 10},
            "stats_del_jugador.tackles": {"$gt": 10},
            "stats_del_jugador.yellow_cards": {"$lt": 5},
            "stats_del_jugador.red_cards": {"$lt": 2}
        }
    },

    {
        "$sort": {"market_value": 1}  # Menor precio primero
    },
    {
        "$limit": 10  # Limitar a los top 10 para tener opciones y no solo uno
    }
]

defensas = list(db.players.aggregate(defensas_pipeline))
for player in defensas:
    print(f"Nombre: {player['name']} | Interceptions: {player['stats_del_jugador']['interceptions']} | Tackles: {player['stats_del_jugador']['tackles']} | Market Value: {player['market_value']}")
if len(defensas) >= 2:
    DFC1 = defensas[0]
    DFC2 = defensas[1]
    print(f"🏆 Defensa seleccionado: {DFC1['name']} | Interceptions: {DFC1['stats_del_jugador']['interceptions']} | Tackles: {DFC1['stats_del_jugador']['tackles']} | Market Value: {DFC1['market_value']}")
    print(f"🏆 Defensa seleccionado: {DFC2['name']} | Interceptions: {DFC2['stats_del_jugador']['interceptions']} | Tackles: {DFC2['stats_del_jugador']['tackles']} | Market Value: {DFC2['market_value']}")
elif len(defensas) == 1:
    DFC1 = defensas[0]
    print(f"🏆 Defensa seleccionado (solo uno disponible): {DFC1['name']} | Interceptions: {DFC1['stats_del_jugador']['interceptions']} | Tackles: {DFC1['stats_del_jugador']['tackles']} | Market Value: {DFC1['market_value']}")
else:
    print("❌ No se encontró defensa que cumpla los criterios")

Nombre: João Aurélio | Interceptions: 22 | Tackles: 33 | Market Value: 49000
Nombre: Bebeto | Interceptions: 21 | Tackles: 59 | Market Value: 53000
Nombre: João Afonso | Interceptions: 20 | Tackles: 12 | Market Value: 53000
Nombre: Veysel Sarı | Interceptions: 19 | Tackles: 30 | Market Value: 97000
Nombre: Allan | Interceptions: 22 | Tackles: 26 | Market Value: 105000
Nombre: Aderllan Santos | Interceptions: 11 | Tackles: 22 | Market Value: 105000
Nombre: Mahamadou Nagida | Interceptions: 13 | Tackles: 15 | Market Value: 145000
Nombre: Anıl Yiğit Çınar | Interceptions: 14 | Tackles: 13 | Market Value: 210000
Nombre: Mawouna Amevor | Interceptions: 31 | Tackles: 22 | Market Value: 210000
Nombre: Denis Odoi | Interceptions: 15 | Tackles: 15 | Market Value: 210000
🏆 Defensa seleccionado: João Aurélio | Interceptions: 22 | Tackles: 33 | Market Value: 49000
🏆 Defensa seleccionado: Bebeto | Interceptions: 21 | Tackles: 59 | Market Value: 53000


In [ ]:
# Seguimos con el lateral izquierdo. Como hemos dicho antes, queremos que nuestro ataques fluyan por la 
# izquierda. Por eso, buscamos un lateral izquierdo que tenga una buena capacidad ofensiva para que pueda 
# sumarse a las jugadas ofensivas y aportar asistencias. Para ello, filtramos por aquellos laterales 
# izquierdos que tengan asistencias, lo que indica que generan oportunidades de gol para sus compañeros. 
# Además, nos interesa que puedan marcar goles, luego que tengan un gran valor de expected goals.

izquierdos_pipeline = [
    {
        "$lookup": {
            "from": "stats",
            "localField": "player_id",
            "foreignField": "player_id",
            "as": "stats_del_jugador"
        }
    },
    {
        "$unwind": "$stats_del_jugador"
    },
    {
        "$match": {
            "position": "D",      
        }
    },

    {
        "$sort": {"stats_del_jugador.assists": -1, "stats_del_jugador.expected_goals": -1 }  # Menor precio primero
    },
    {
        "$limit": 10  # Limitar a los top 10 para tener opciones y no solo uno
    }
]

izquierdos = list(db.players.aggregate(izquierdos_pipeline))
for player in izquierdos:
    print(f"Nombre: {player['name']} | Expected Goals: {player['stats_del_jugador']['expected_goals']} | Assists: {player['stats_del_jugador']['assists']} | Market Value: {player['market_value']}")
if izquierdos:
    LI = izquierdos[0]
    print(f"🏆 Lateral izquierdo seleccionado: {LI['name']} | Expected Goals: {LI['stats_del_jugador']['expected_goals']} | Assists: {LI['stats_del_jugador']['assists']} | Market Value: {LI['market_value']}")
else:
    print("❌ No se encontró lateral izquierdo que cumpla los criterios")

Nombre: Julian Ryerson | Expected Goals: 0.5824 | Assists: 12 | Market Value: 18900000
Nombre: Souffian El Karouani | Expected Goals: 1.7044 | Assists: 10 | Market Value: 14000000
Nombre: Mauro Júnior | Expected Goals: 1.1998 | Assists: 8 | Market Value: 17600000
Nombre: Matthieu Udol | Expected Goals: 3.8302 | Assists: 7 | Market Value: 4800000
Nombre: Jordan Bos | Expected Goals: 1.9398 | Assists: 7 | Market Value: 7400000
Nombre: David Raum | Expected Goals: 2.0733 | Assists: 6 | Market Value: 22000000
Nombre: Alberto Costa | Expected Goals: 0.9074 | Assists: 6 | Market Value: 14000000
Nombre: Dinis Pinto | Expected Goals: 0.8107 | Assists: 6 | Market Value: 3300000
Nombre: Vladimír Coufal | Expected Goals: 0.6957 | Assists: 6 | Market Value: 2600000
Nombre: Jurriën Timber | Expected Goals: 4.7029 | Assists: 5 | Market Value: 73000000
🏆 Lateral izquierdo seleccionado: Julian Ryerson | Expected Goals: 0.5824 | Assists: 12 | Market Value: 18900000


In [14]:
# Ahora, queremos los medioscentros. Para esta posición, nos interesa que tengan una buena 
# capacidad de pase para asegurar la creación de juego en el centro del campo. También nos 
# interesa que tengan un buen número de intercepciones para asegurar que recuperan balones y 
# ayudan en defensa. 
mediocentros_pipeline = [
    {
        "$lookup": {
            "from": "stats",
            "localField": "player_id",
            "foreignField": "player_id",
            "as": "stats_del_jugador"
        }
    },
    {
        "$unwind": "$stats_del_jugador"
    },
    {
        "$match": {
            "position": "M",
            "stats_del_jugador.interceptions": {"$gte": 5},  # Filtrar por intercepciones mínimas
            "stats_del_jugador.assists": {"$gte": 5}  # Filtrar por asistencias mínimas
        },
    },

    {
        "$sort": {"stats_del_jugador.interceptions": -1, "stats_del_jugador.assists": -1}   
    },
    {
        "$limit": 10  # Limitar a los top 10 para tener opciones y no solo uno
    }
]

mediocentros = list(db.players.aggregate(mediocentros_pipeline))
for player in mediocentros:
    print(f"Nombre: {player['name']} | Interceptions: {player['stats_del_jugador']['interceptions']} | Assists: {player['stats_del_jugador']['assists']} | Market Value: {player['market_value']}")

if mediocentros:
    MC1 = mediocentros[0]
    print(f"🏆 Mediocentro seleccionado: {MC1['name']} | Interceptions: {MC1['stats_del_jugador']['interceptions']} | Assists: {MC1['stats_del_jugador']['assists']} | Market Value: {MC1['market_value']}")
    
else:
    print("❌ No se encontró mediocentro que cumpla los criterios")

Nombre: James Garner | Interceptions: 50 | Assists: 6 | Market Value: 37000000
Nombre: Kodai Sano | Interceptions: 37 | Assists: 7 | Market Value: 12900000
Nombre: Jonathan Clauss | Interceptions: 35 | Assists: 6 | Market Value: 4099999
Nombre: Joris van Overeem | Interceptions: 32 | Assists: 7 | Market Value: 850000
Nombre: Jordan Holsgrove | Interceptions: 30 | Assists: 7 | Market Value: 3700000
Nombre: Declan Rice | Interceptions: 30 | Assists: 5 | Market Value: 114000000
Nombre: Jeff Hardeveld | Interceptions: 29 | Assists: 6 | Market Value: 435000
Nombre: Luis Milla | Interceptions: 28 | Assists: 9 | Market Value: 3200000
Nombre: Angelo Stiller | Interceptions: 26 | Assists: 5 | Market Value: 46000000
Nombre: Vitinha | Interceptions: 25 | Assists: 7 | Market Value: 102000000
🏆 Mediocentro seleccionado: James Garner | Interceptions: 50 | Assists: 6 | Market Value: 37000000


Aquí se nos abre un amplio abanico de posibilidades, pero para mantener el presupuesto del club e invertir en la liga española, seleccionaremos al jugador que nos indica el algoritmo y a otro para que le acompañe en el pivote. Somos unos románticos de Luis Milla y es justo lo que buscamos: Bueno, Bonito y Barato.

In [15]:
query = { "name" : "Luis Milla", "position" : "M" }
MC2 = db.players.find_one(query)
print(f"🏆 Segundo mediocentro seleccionado: {MC2['name']}") 

🏆 Segundo mediocentro seleccionado: Luis Milla


In [16]:
# Ahora, queremos el mediocentro ofensivo. Para esta posición, nos interesa que tengan una buena 
# capacidad de pase para asegurar la creación de juego en el centro del campo. Debido a su perfil 
# ofensivo, nos interesa que tengan un buen número de asistencias para asegurar que generan 
# oportunidades de gol para sus compañeros. Además, nos interesa que tengan goles para contribuir 
# desde segunda línea.

mco_pipeline = [
    {
        "$lookup": {
            "from": "stats",
            "localField": "player_id",
            "foreignField": "player_id",
            "as": "stats_del_jugador"
        }
    },
    {
        "$unwind": "$stats_del_jugador"
    },
    {
        "$match": {
            "position": "M",
            "stats_del_jugador.goals": {"$gte": 5},  # Filtrar por goles mínimos
            "stats_del_jugador.assists": {"$gte": 5}  # Filtrar por asistencias mínimas
        },
    },

    {
        "$sort": {"stats_del_jugador.assists": -1, "stats_del_jugador.goals": -1}   
    },
    {
        "$limit": 10  # Limitar a los top 10 para tener opciones y no solo uno
    }
]

mco = list(db.players.aggregate(mco_pipeline))
for player in mco:
    print(f"Nombre: {player['name']} | Goals: {player['stats_del_jugador']['goals']} | Assists: {player['stats_del_jugador']['assists']} | Market Value: {player['market_value']}")

if mco:
    MCO = mco[0]
    print(f"🏆 Mediocentro seleccionado: {MCO['name']} | Goals: {MCO['stats_del_jugador']['goals']} | Assists: {MCO['stats_del_jugador']['assists']} | Market Value: {MCO['market_value']}")
    
else:
    print("❌ No se encontró mediocentro que cumpla los criterios")

Nombre: Michael Olise | Goals: 11 | Assists: 17 | Market Value: 125000000
Nombre: Bruno Fernandes | Goals: 8 | Assists: 16 | Market Value: 37000000
Nombre: Federico Dimarco | Goals: 6 | Assists: 14 | Market Value: 48000000
Nombre: Joey Veerman | Goals: 8 | Assists: 13 | Market Value: 28000000
Nombre: Marco Asensio | Goals: 11 | Assists: 12 | Market Value: 15500000
Nombre: Luis Díaz | Goals: 15 | Assists: 11 | Market Value: 76000000
Nombre: Mohamed Ihattaren | Goals: 6 | Assists: 10 | Market Value: 3200000
Nombre: Lamine Yamal | Goals: 14 | Assists: 9 | Market Value: 215000000
Nombre: Barış Alper Yılmaz | Goals: 6 | Assists: 9 | Market Value: 28000000
Nombre: Ivan Perišić | Goals: 5 | Assists: 9 | Market Value: 1300000
🏆 Mediocentro seleccionado: Michael Olise | Goals: 11 | Assists: 17 | Market Value: 125000000


In [ ]:
# Finalmente, vamos a por la línea de ataque. Para esta posición, 
# nos interesa que tengan una buena capacidad de finalización para 
# asegurar que marcan goles. Nos fijaremos en los goles y los tiros totales.

atacantes_pipeline = [
    {
        "$lookup": {
            "from": "stats",
            "localField": "player_id",
            "foreignField": "player_id",
            "as": "stats_del_jugador"
        }
    },
    {
        "$unwind": "$stats_del_jugador"
    },
    {
        "$match": {
            "position": "F",
            "stats_del_jugador.goals": {"$gte": 5},  # Filtrar por goles mínimos
            "stats_del_jugador.total_shots": {"$gte": 35}  # Filtrar por tiros totales mínimos
        },
    },

    {
        "$sort": {"stats_del_jugador.total_shots": -1, "stats_del_jugador.goals": -1}   
    },
    {
        "$limit": 10  # Limitar a los top 10 para tener opciones y no solo uno
    }
]

atacantes = list(db.players.aggregate(atacantes_pipeline))
for player in atacantes:
    print(f"Nombre: {player['name']} | Goals: {player['stats_del_jugador']['goals']} | Assists: {player['stats_del_jugador']['assists']} | Market Value: {player['market_value']}")
    
if not atacantes:
    print("❌ No se encontró atacante que cumpla los criterios")

Nombre: Kylian Mbappé | Goals: 23 | Assists: 4 | Market Value: 212000000
Nombre: Harry Kane | Goals: 31 | Assists: 5 | Market Value: 71000000
Nombre: Luis Javier Suárez | Goals: 24 | Assists: 4 | Market Value: 19900000
Nombre: Erling Haaland | Goals: 22 | Assists: 7 | Market Value: 218000000
Nombre: Deniz Undav | Goals: 18 | Assists: 5 | Market Value: 18500000
Nombre: Moise Kean | Goals: 8 | Assists: 1 | Market Value: 46000000
Nombre: Mason Greenwood | Goals: 15 | Assists: 4 | Market Value: 49000000
Nombre: Ayase Ueda | Goals: 22 | Assists: 1 | Market Value: 14600000
Nombre: Vedat Muriqi | Goals: 18 | Assists: 1 | Market Value: 4400000
Nombre: Kenan Yıldız | Goals: 10 | Assists: 6 | Market Value: 72000000


No tenemos fairplay para robar al Madrid a Mbappe, por lo que necesitamos un plan B. Iremos a por Harry Kane, quien está por renovar en el Bayern y podemos seducirle con mejorar su perfil. Luego, apostaremos por talento joven para el extremo, así que le robaremos a la Juventus a Kenan Yildiz.

In [18]:
if atacantes:
    ST = atacantes[1]
    EI = atacantes[9]
    print(f"🏆 Atacante seleccionado: {ST['name']} | Goals: {ST['stats_del_jugador']['goals']} | Assists: {ST['stats_del_jugador']['assists']} | Market Value: {ST['market_value']}")
    print(f"🏆 Atacante seleccionado: {EI['name']} | Goals: {EI['stats_del_jugador']['goals']} | Assists: {EI['stats_del_jugador']['assists']} | Market Value: {EI['market_value']}")

🏆 Atacante seleccionado: Harry Kane | Goals: 31 | Assists: 5 | Market Value: 71000000
🏆 Atacante seleccionado: Kenan Yıldız | Goals: 10 | Assists: 6 | Market Value: 72000000


Ahora, iremos en busca de nuestro jugador estrella. Queremos un jugador para el extremo derecho, intentar que sea joven para tener marketing, con alto rating y goles. Es el  jugador para el que  tenemos disponibles más de 100M €.

In [19]:
# Finalmente, vamos  a por la línea de ataque. Para esta posición, 
# nos interesa que tengan una buena capacidad de finalización para 
# asegurar que marcan goles. Nos fijaremos en los goles y los tiros totales.

estrella_pipeline = [
    {
        "$lookup": {
            "from": "stats",
            "localField": "player_id",
            "foreignField": "player_id",
            "as": "stats_del_jugador"
        }
    },
    {
        "$unwind": "$stats_del_jugador"
    },
    {
        "$match": {
            "position": "F",
            "market_value": {"$gte": 90000000},  # Filtrar por valor de mercado para asegurar que es una estrella
            "stats_del_jugador.goals": {"$gte": 5},  # Filtrar por goles mínimos
        },
    },

    {
        "$sort": {"stats_del_jugador.rating": -1, "stats_del_jugador.market_value": -1}   
    },
    {
        "$limit": 10  # Limitar a los top 10 para tener opciones y no solo uno
    }
]

estrellas = list(db.players.aggregate(estrella_pipeline))
for player in estrellas:
    print(f"Nombre: {player['name']} | Goals: {player['stats_del_jugador']['goals']} | Rating: {player['stats_del_jugador']['rating']} | Market Value: {player['market_value']}")
    
if not estrellas:
    print("❌ No se encontró estrella que cumpla los criterios")

Nombre: Kylian Mbappé | Goals: 23 | Rating: 7.7708333333333 | Market Value: 212000000
Nombre: Ousmane Dembélé | Goals: 8 | Rating: 7.49375 | Market Value: 97000000
Nombre: Vinicius Júnior | Goals: 11 | Rating: 7.3857142857143 | Market Value: 156000000
Nombre: Erling Haaland | Goals: 22 | Rating: 7.351724137931 | Market Value: 218000000
Nombre: Bukayo Saka | Goals: 6 | Rating: 7.2148148148148 | Market Value: 126000000
Nombre: Désiré Doué | Goals: 5 | Rating: 7.0470588235294 | Market Value: 93000000
Nombre: Julián Alvarez | Goals: 8 | Rating: 6.9551724137931 | Market Value: 97000000


Por el mismo caso que antes, no podemos quitar a Mbappé al Madrid. Nos quedaremos con el último Balón de Oro, el cual generará marketing en el club y proveerá de asistencias a nuestro delantero Harry Kane. 

In [20]:
if estrellas:
    ED = estrellas[1]
    print(f"🏆 Estrella seleccionada: {ED['name']} | Goals: {ED['stats_del_jugador']['goals']} | Assists: {ED['stats_del_jugador']['assists']} | Market Value: {ED['market_value']} | Rating: {ED['stats_del_jugador']['rating']} ")

🏆 Estrella seleccionada: Ousmane Dembélé | Goals: 8 | Assists: 5 | Market Value: 97000000 | Rating: 7.49375 


Crearemos una nueva coleccción en la que incluiremos nuestro equipo titular y así, podremos ir viendo sus estadísticas y el valor global del equipo

In [21]:
db["equipo_titular"].insert_many([GK, LD, DFC1, DFC2, LI, MC1, MC2, MCO, ST, EI, ED])
print("✅ Equipo titular insertado en la colección 'equipo_titular'.")

✅ Equipo titular insertado en la colección 'equipo_titular'.


In [22]:
for player in db.equipo_titular.find():
    print(f"Nombre: {player['name']} | Position: {player['position']} | Market Value: {player['market_value']}")

Nombre: Kayne van Oevelen | Position: G | Market Value: 2100000
Nombre: Ivo Pinto | Position: D | Market Value: 260000
Nombre: João Aurélio | Position: D | Market Value: 49000
Nombre: Bebeto | Position: D | Market Value: 53000
Nombre: Julian Ryerson | Position: D | Market Value: 18900000
Nombre: James Garner | Position: M | Market Value: 37000000
Nombre: Luis Milla | Position: M | Market Value: 3200000
Nombre: Michael Olise | Position: M | Market Value: 125000000
Nombre: Harry Kane | Position: F | Market Value: 71000000
Nombre: Kenan Yıldız | Position: F | Market Value: 72000000
Nombre: Ousmane Dembélé | Position: F | Market Value: 97000000


Calculemos el valor global del equipo para ver si no hemos superado el presupuesto límite que teníamos:

In [23]:
precios_total = [
    {
        "$group": {
            "_id": None,
            "total_price": { "$sum": "$market_value" }
        }
    }
]

GASTO_TOTAL_FICHAJES = list(db.equipo_titular.aggregate(precios_total))
print(f"Gasto total en fichajes: {GASTO_TOTAL_FICHAJES[0]['total_price']} €")

Gasto total en fichajes: 426562000 €


Por tanto, como teníamos un presupuesto de 500M €, hemos cumplido con la planificación de compras, también cumpliendo el objetivo de haber gastado menos en nuestro jugador estrella de lo que teníamos presupuestado inicialmente, mostrando el poder de negociación.

Sin embargo, nos llama nuestro agente y nos dice que el precio de uno de los defensas seleccionados está complicado de conseguir ya que es figura en su equipo, y no le dejarán  salir por menos de su cláusula. Por tanto, tendremos que actualizar el precio y ver si seguimos dentro de nuestro presupuesto.

In [24]:
# Actualizar el market_value del jugador Bebeto
CLAUSULA = 5000000

db.equipo_titular.update_one(
    {"name": "Bebeto"},
    {"$set": {"market_value": CLAUSULA}}
)

print("✅ Market value de Bebeto actualizado.")

✅ Market value de Bebeto actualizado.


Luego, el valor del equipo actual queda de la siguiente manera:

In [25]:
precios_total_actualizado = [
    {
        "$group": {
            "_id": None,
            "total_price": { "$sum": "$market_value" }
        }
    }
]

GASTO_TOTAL_FICHAJES = list(db.equipo_titular.aggregate(precios_total_actualizado))
print(f"Gasto total en fichajes: {GASTO_TOTAL_FICHAJES[0]['total_price']} €")

Gasto total en fichajes: 431509000 €


In [26]:
# Filtrar defensas (position = "D") y proyectar solo name, position y market_value
pipeline_defensas = [
    {"$match": {"position": "D"}},
    {"$project": {"name": 1, "position": 1, "market_value": 1, "_id": 0}}
]

precio_defensas = [
    {
        "$match": {"position": "D"}
    },
    {
        "$group": {
            "_id": None,
            "total_price_defensas": { "$sum": "$market_value" }
        }
    }
]

defensas = list(db.equipo_titular.aggregate(pipeline_defensas))
valor_defensa = list(db.equipo_titular.aggregate(precio_defensas))

for defensa in defensas:
    print(f"Nombre: {defensa['name']} | Posición: {defensa['position']} | Valor: {defensa['market_value']} €")

print(f"Gasto total en fichajes de defensas: {valor_defensa[0]['total_price_defensas']} €")

Nombre: Ivo Pinto | Posición: D | Valor: 260000 €
Nombre: João Aurélio | Posición: D | Valor: 49000 €
Nombre: Bebeto | Posición: D | Valor: 5000000 €
Nombre: Julian Ryerson | Posición: D | Valor: 18900000 €
Gasto total en fichajes de defensas: 24209000 €


Luego, hemos conseguido una defensa barata a pesar de tener que pagar la cláusula de Bebeto. En suma, hemos cumplido con creces la planificación de fichajes, ahorrando 70M € al club, destinados a futuros fichajes.